### Остаточные нейронные сети (ResNet):
Темы:
1. **Проблему затухающих градиентов** и почему увеличение слоев не всегда хорошо.

2. **Ключевую идею ResNet** – обходные связи (`skip connections`), которые совершили "революцию глубины".

3. **Два типа блоков**:

- `BasicBlock` для ResNet‑18/34

- `Bottleneck` для ResNet‑50/101/152

4. **Как устроены архитектуры ResNet‑18 и ResNet‑50** на уровне слоёв.

5. **Пример кода**: загрузка предобученной модели и классификация своего изображения.

### 1. Введение: почему не работают «очень глубокие» сети?

Сначала интуиция была простой: чем больше слоёв, тем лучше сеть обучается. Взяли VGG‑16 (16 слоёв) – работает. Сделали VGG‑19 (19 слоёв) – ещё лучше. Логично предположить, что VGG‑34 (34 слоя) превзойдёт VGG‑18. Но на практике сеть с 34 слоями **обучалась хуже**, чем её же 18-слойная версия.

#### Почему так происходит?
**Проблема затухающих (исчезающих) градиентов** (`vanishing gradients`):

- При обратном распространении ошибки градиенты становятся очень маленькими к начальным слоям.

- Последние слои учатся быстро, их ошибка стремится к нулю, и градиент для начальных слоев блокируется.

- Сеть "застревает" далеко от оптимума.

#### Как долго решали проблему?

- **Предобучение фрагментов** (разбивали сеть на части и обучали их отдельно без учителя – метод Хинтона).

- **Улучшенная инициализация весов** (работы Ян ЛеКуна, Глоро и др.).

- **Batch Normalization** (Иоффе и Сегеди, 2015) – ускоряет сходимость, но полностью проблему не снимает.

Настоящий прорыв случился в 2015 году, когда **Microsoft Research** (Каймин Хэ и коллеги) предложили **остаточные сети (Residual Networks, ResNet)**

### 2. Идея ResNet

#### Базовая идея
Вместо того чтобы заставлять несколько свёрточных слоёв выучить сложную функцию $H(x)$, мы заставляем их выучить **остаток (residual)** $F(x) = H(x) - x$.
$$
H(x) = F(x) + x
$$

Блок, который вычисляет $F(x)$ и потом прибавляет исходный $x$, называется **остаточным блоком**.

#### Структура простейшего Basic Block

```
        x
        |
     Conv 3x3, BN, ReLU
        |
     Conv 3x3, BN
        |
        +  ←─┐
        |    |
       ReLU  x  (обходной путь)
```

- Обходной путь (`skip connection`) добавляет вход $x$ напрямую к выходу свёрток.

- Таким образом, свёртки должны выучить лишь **поправку** $F(x)$, а не всю функцию $H(x)$.

- Это намного проще с точки зрения обучения.

#### Почему это решает проблему градиентов?

Для такого блока градиент на обратном проходе:

$$
\frac{\partial \text{out}}{\partial x} = \frac{\partial (F(x) + x)}{\partial x} = F'(x) + 1
$$

Благодаря **единице** градиент не может полностью затухнуть, даже если $F'(x)$ маленький. Это позволяет обучать сети с **сотнями и тысячами слоёв**.

### 3. Два типа остаточных блоков

В ResNet используются два типа блоков: **Basic** (для «лёгких» версий) и **Bottleneck** (для глубоких моделей).

#### 3.1 Basic Block (используется в ResNet‑18 и ResNet‑34)

**Структура**:

- Два свёрточных слоя 3×3, каждый с BatchNorm и ReLU (кроме последнего перед суммированием).

- Количество каналов на входе и выходе одинаковое (если нет изменения размерности).

- Если нужно уменьшить размер карт признаков – в первом свёрточном слое ставим `stride=2`, а в обходном пути добавляем свёртку 1×1 `stride=2`, чтобы согласовать размеры\.

In [2]:
from torch import nn

class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_channels=64, out_channels=64, stride=1, downsample=None):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3,stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_channels)
        self.relu  = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_channels)
        self.downsample = downsample  # Свёртка 1×1 для согласования размеров

    def forward(self, x):
        identity = x
        if self.downsample is not None:
            identity = self.downsample(x)
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out += identity
        out = self.relu(out)
        return out

model = BasicBlock()
print(model.parameters)

<bound method Module.parameters of BasicBlock(
  (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
)>


#### 3.2 Bottleneck Block (используется в ResNet‑50, ResNet‑101, ResNet‑152)

**Почему Bottleneck?** Он в 3 раза эффективнее по числу операций!

Структура:
```
    in_channels = 256
            ↓
     Conv 1×1, 64        ← сжимаем каналы в 4 раза
       BN, ReLU
            ↓
     Conv 3×3, 64        ← основной слой (с stride, если надо)
       BN, ReLU
            ↓
     Conv 1×1, 256       ← расширяем обратно до 256 каналов
       BN
            ↓
            +  ←─┐
            |    |
          ReLU  x (с возможной downsampling)
```

Подсчёт операций для входного тензора со 128 каналами:

- Если бы был один слой 3×3 с 128 входами и 128 выходами: $3×3×128×128 = 147\,456$ операций.

- Bottleneck: $1×1×128×64 + 3×3×64×64 + 1×1×64×128 = 8192 + 36\,864 + 8192 = 53\,248$ операций.

**Выигрыш**: почти в **3 раза** меньше вычислений при сопоставимом качестве!

>**Важно**: В реальных ResNet‑50 параметр `expansion = 4`. Входные каналы расширяются в 4 раза на выходе блока.

In [ ]:
class Bottleneck(nn.Module):
    expansion = 4

    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        super().__init__()
        # 1×1 сжатие
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)

        # 3×3 основной слой
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        # 1×1 расширение
        self.conv3 = nn.Conv2d(out_channels, out_channels * self.expansion, kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(out_channels * self.expansion)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample

    def forward(self, x):
        identity = x
        if self.downsample is not None:
            identity = self.downsample(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)
        out = self.conv3(out)
        out = self.bn3(out)

        out += identity
        out = self.relu(out)
        return out

model = BasicBlock()
print(model.parameters)

### 4. Архитектура ResNet‑18

#### Общий вид
```
    Изображение 224×224×3
        ↓
    Conv 7×7, 64, stride=2
    MaxPool 3×3, stride=2
        ↓
    Layer1: 2×BasicBlock (64 → 64)
    Layer2: 2×BasicBlock (64 → 128, downsampling)   ← размер карт /2, каналов *2
    Layer3: 2×BasicBlock (128 → 256)                ← размер карт /4
    Layer4: 2×BasicBlock (256 → 512)                ← размер карт /8
        ↓
    AdaptiveAvgPool2d → (512, 1, 1)
        ↓
    FC (512 → 1000)
```

**Детали Layer2** (там, где меняется размерность):

- Вход: `(64, H, W)`

- Первый блок содержит `stride=2` в первом свёрточном слое → выход `(128, H/2, W/2)`.

- Поскольку размеры изменились, в `downsample` добавляется свёртка 1×1 `stride=2`.

- Второй блок Layer2 уже без `downsample`, просто два свёрточных слоя 3×3.

In [3]:
from torchvision import models

model_resnet18 = models.resnet18(pretrained=True)   # предобученная на ImageNet
print(model_resnet18)

c:\Users\mrkee\AppData\Local\Programs\Python\Python314\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\mrkee\AppData\Local\Programs\Python\Python314\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\mrkee/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100.0%


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta

В выводе увидим те самые `Layer1`, `Layer2`, `Layer3`, `Layer4`, каждый из которых содержит `BasicBlock`

### 5. Архитектура ResNet‑50

#### Распределение Bottleneck блоков по слоям:
| Слой     | Количество Bottleneck блоков |
|----------|------------------------------|
| Layer1   | 3                            |
| Layer2   | 4                            |
| Layer3   | 6                            |
| Layer4   | 3                            |

**Что происходит внутри Layer2 (первый блок со сменой размерности)**:

- Вход: `(256, H, W)`   (256 = 64×4, потому что `expansion=4`)

- Первый Bottleneck блока Layer2:

  - Conv1 (1×1): 256 → 128 (сжимаем каналы)

  - Conv2 (3×3): `stride=2` → карты признаков уменьшаются: `(128, H/2, W/2)`

  - Conv3 (1×1): 128 → 512 (расширяем до 128×4)

  - В обходном пути добавляется `downsample` (свёртка 1×1 `stride=2`), чтобы вход `(256, H, W)` превратить в `(512, H/2, W/2)`.

- Последующие 3 блока Layer2 – без `downsample`, внутри у них всё идентично.

**Выход Layer4**: `(512×4, H/8, W/8) = (2048, H/8, W/8)`. Затем – AdaptiveAvgPool2d и полносвязный слой на 1000 классов[reference:16].

In [4]:
import torch
import torchvision.transforms as transforms
from PIL import Image
from torchvision import models

# 1. Загружаем предобученную модель
weights = models.ResNet50_Weights.DEFAULT
model = models.resnet50(weights=weights)
model.eval()

# 2. Преобразования для изображения: привести к 224x224
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# 3. Загружаем метки классов ImageNet
cats = weights.meta['categories']

# 4. Открываем изображение, преобразуем и добавляем batch dimension
img = Image.open("datasets/cnn/car.jpg").convert('RGB')
img_tensor = transform(img).unsqueeze(0)  # [1, 3, 224, 224]

# 5. Инференс
with torch.no_grad():
    outputs = model(img_tensor).squeeze()
    probabilities = outputs.softmax(dim=0).sort(descending=True)

# 6. Топ‑5 предсказаний
for s, i in zip(probabilities[0][:5], probabilities[1][:5]):
    print(f"{cats[i]}: {s:.4f}")

beach wagon: 0.1656
car wheel: 0.0871
sports car: 0.0705
grille: 0.0542
convertible: 0.0530


> Файл `imagenet_classes.txt` нужно скачать отдельно. Готовую версию можно взять из официального репозитория PyTorch.

**Что означают `unsqueeze(0)` и `squeeze()`?**

- `unsqueeze(0)` добавляет фиктивное измерение батча: было `(3, H, W)`, стало `(1, 3, H, W)` – модель ожидает именно батч.
- Если мы получили `outputs` формы `(1, 1000)`, то `squeeze()` уберёт единичное измерение, превратив в `(1000,)` для удобной работы с `softmax`.